# Parallel allocation experiment: 1,000 people, 100 models

Compare **LDA-exp**, **LDA-tru**, and **OPT** on the same sampled cohorts using
`src.allocations_parallel`. Each worker evaluates one model. The default is
four workers, 100 candidate models, a cohort size of 1,000, a 10% selection rate
(100 selected), and one cohort per seed in the saved prediction file.

- **LDA-exp:** DP using the model's expected needs, rounded once to integers
  with `np.rint` (halfway values round to even).
- **LDA-tru:** DP using integer true needs.
- **OPT:** the model's top 100 qualification probabilities.

All algorithms are evaluated using **true needs** for disparity. Predicted
utility is the selected people's average qualification probability; realized
utility is their average binary qualification label. The existing validation
loss rule chooses the Rashomon models from the 100 candidates in each split.
Each model retains its own predictions; models share the cohort.

Run the cells in order. Completed results are saved in a separate folder under
`data/results/allocations/parallel_baseline/` and reused on subsequent runs
after file-integrity checks. This notebook does not overwrite the older
baseline experiment files. The comparison table and disparity plot are below.

In [ ]:
import hashlib
import importlib
import json
import platform
import sys
import uuid
from ast import literal_eval
from pathlib import Path
from time import perf_counter

import joblib
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display

# Works from the repository root or test/; the final path is a local fallback.
_candidates = [Path.cwd(), *Path.cwd().parents,
               Path('/Users/kk985/PhD/Research/LessDiscriminatoryAllocations')]
ROOT = next((p for p in _candidates
             if (p / 'src/allocations_parallel.py').is_file()
             and (p / 'data/predictions').is_dir()), None)
if ROOT is None:
    raise FileNotFoundError('Open this notebook from the research repository or its test/ folder.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import src.allocations_parallel
if Path(src.allocations_parallel.__file__).resolve() != (ROOT / 'src/allocations_parallel.py').resolve():
    raise RuntimeError('A different repository is already imported. Restart the kernel.')
importlib.reload(src.allocations_parallel)
from src.allocations_parallel import Allocations

print('Repository:', ROOT)

## Settings

`N_MODELS=100` means 100 **candidate** models; fewer may pass the Rashomon
threshold. The next section shows the eligible count for each seed.
`ITERS_PER_SEED=1` follows the current baseline runner; increase it for more
cohorts and a better estimate of variability. Each extra iteration increases
runtime. Keep `N_JOBS=4` initially; `-1` uses all CPUs and more memory.

The default prediction variant is `average` (expected need from the class
probabilities). Add `"argmax"` to `PRED_METHODS` to compare the saved argmax
predictions as a separate experiment. No models are trained in this notebook.

In [ ]:
METHOD = 'bootstrap'
MODEL_CLASS = 'nn_multiclass'
PRED_METHODS = ['average']
NEED_THRESHOLDS = [1]
SELECTION_RATES = [0.1]

SAMPLE_SIZE = 1000
N_MODELS = 100
ITERS_PER_SEED = 1
N_JOBS = 4
SHOW_PROGRESS = True

RASHOMON_EPS = 0.01
ALLOCATION_EPS = 0.1
NEED_COL = 'gagne_sum_t'
EXP_NEED_COL = 'exp_need_{model}'
GROUP_COL = 'race'
ALGORITHMS = ['OPT', 'LDA-tru', 'LDA-exp']
ALGORITHM_ORDER = ['LDA-exp', 'LDA-tru', 'OPT']

PREDICTION_DIR = ROOT / 'data/predictions'
RAW_DATA_FILE = ROOT / 'data/obermeyer/obermeyer_data_cleaned.csv'
OUTPUT_ROOT = ROOT / 'data/results/allocations/parallel_baseline'

In [ ]:
def file_hash(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()


def save_json(value, path):
    path = Path(path)
    temporary = path.with_name(path.name + '.' + uuid.uuid4().hex + '.tmp')
    temporary.write_text(json.dumps(value, indent=2, sort_keys=True) + '\n')
    temporary.replace(path)


def save_csv(frame, path):
    path = Path(path)
    temporary = path.with_name(path.name + '.' + uuid.uuid4().hex + '.tmp')
    frame.to_csv(temporary, index=False)
    temporary.replace(path)


def make_allocator(frame, selection_rate):
    return Allocations(
        df=frame, sample_size=SAMPLE_SIZE, selection_rate=selection_rate,
        qualification_col='y', need_col=NEED_COL, group_col=GROUP_COL,
        rashomon_eps=RASHOMON_EPS, allocation_eps=ALLOCATION_EPS,
        n_models=N_MODELS, iters_per_seed=ITERS_PER_SEED,
        n_jobs=N_JOBS, show_progress=SHOW_PROGRESS,
    )


def allocation_tag(pred_method, need_threshold, selection_rate):
    return (f'{METHOD}_{MODEL_CLASS}_{pred_method}_y-qualified_gagne_{need_threshold}'
            f'_rate-{selection_rate}_size-{SAMPLE_SIZE}')


def read_result_files(paths):
    result = {
        key: pd.read_csv(paths[key], converters={'selected': literal_eval, 'models': literal_eval})
        for key in ('OPT', 'LDA-tru', 'LDA-exp')
    }
    result['people'] = pd.read_csv(paths['people'], converters={'people': literal_eval})
    result['diagnostics'] = pd.read_csv(paths['diagnostics'])
    return result

## Load predictions and check the run

Input fingerprints, settings, library versions, and the parallel DP source
fingerprint identify the output folder. Changing any of these creates a
different folder. The original random cohort sampler is retained: group
sizes vary, and all algorithms use identical people within each cohort.

In [ ]:
if not PRED_METHODS or not NEED_THRESHOLDS or not SELECTION_RATES:
    raise ValueError('Provide at least one prediction variant, threshold, and selection rate.')
if any(len(items) != len(set(items)) for items in (PRED_METHODS, NEED_THRESHOLDS, SELECTION_RATES)):
    raise ValueError('Experiment settings must not contain duplicate values.')
if not set(PRED_METHODS) <= {'average', 'argmax'}:
    raise ValueError('PRED_METHODS must contain average and/or argmax.')

raw_groups = pd.read_csv(RAW_DATA_FILE, usecols=['person_id', GROUP_COL])
if not raw_groups.person_id.is_unique:
    raise ValueError('The raw data must have unique person IDs.')
model_columns = [f'm_{i}' for i in range(1, N_MODELS + 1)]
need_columns = [Allocations.exp_need_col_for_model(model, EXP_NEED_COL) for model in model_columns]
required_columns = ['person_id', 'seed', 'y', NEED_COL, *model_columns, *need_columns]
prediction_cache, input_fingerprints, eligible_models, model_count_rows = {}, {}, {}, []

for pred_method in PRED_METHODS:
    for threshold in NEED_THRESHOLDS:
        path = PREDICTION_DIR / f'test_{METHOD}_{MODEL_CLASS}_{pred_method}_qualified_gagne_{threshold}.csv'
        frame = pd.read_csv(path, usecols=required_columns)
        frame = frame.merge(raw_groups, on='person_id', how='left', validate='many_to_one')
        population = frame.loc[frame.person_id >= 0]
        if not population[GROUP_COL].isin([0, 1]).all():
            raise ValueError(f'{path.name}: some people have missing or invalid group labels.')
        if frame.seed.isna().any() or population.duplicated(['seed', 'person_id']).any():
            raise ValueError(f'{path.name}: missing seeds or duplicate people within a split.')
        if not population.y.isin([0, 1]).all():
            raise ValueError(f'{path.name}: qualification labels must be binary.')
        actual_needs = population[NEED_COL].to_numpy(dtype=float)
        if (not np.isfinite(actual_needs).all() or (actual_needs < 0).any()
                or (actual_needs != np.floor(actual_needs)).any()):
            raise ValueError(f'{path.name}: true needs must be finite nonnegative integers.')
        prediction_cache[(pred_method, threshold)] = frame
        input_fingerprints[str(path.resolve())] = file_hash(path)
        check = make_allocator(frame, SELECTION_RATES[0])
        for seed in sorted(frame.seed.unique()):
            models = check.rashomon_models(frame.loc[(frame.seed == seed) & (frame.person_id < 0)])
            eligible_models[(pred_method, threshold, seed)] = models
            model_count_rows.append({'pred_variant': pred_method, 'need_threshold': threshold,
                                     'seed': seed, 'candidate_models': N_MODELS,
                                     'eligible_models': len(models), 'cohorts': ITERS_PER_SEED})

run_config = {
    'schema_version': 1, 'method': METHOD, 'model_class': MODEL_CLASS,
    'pred_methods': PRED_METHODS, 'need_thresholds': NEED_THRESHOLDS,
    'selection_rates': SELECTION_RATES, 'sample_size': SAMPLE_SIZE,
    'n_models': N_MODELS, 'iters_per_seed': ITERS_PER_SEED, 'n_jobs': N_JOBS,
    'rashomon_eps': RASHOMON_EPS, 'allocation_eps': ALLOCATION_EPS,
    'need_col': NEED_COL, 'exp_need_col': EXP_NEED_COL, 'group_col': GROUP_COL,
    'algorithms': ALGORITHMS, 'prediction_sha256': input_fingerprints,
    'raw_data_sha256': file_hash(RAW_DATA_FILE),
    'parallel_source_sha256': file_hash(ROOT / 'src/allocations_parallel.py'),
    'versions': {'python': platform.python_version(), 'numpy': np.__version__,
                 'pandas': pd.__version__, 'joblib': joblib.__version__},
}
run_id = hashlib.sha256(json.dumps(run_config, sort_keys=True).encode()).hexdigest()[:12]
RUN_DIR = OUTPUT_ROOT / f'size-{SAMPLE_SIZE}_models-{N_MODELS}_iters-{ITERS_PER_SEED}_{run_id}'
RUN_DIR.mkdir(parents=True, exist_ok=True)
config_path = RUN_DIR / 'run_config.json'
if config_path.exists() and json.loads(config_path.read_text()) != run_config:
    raise ValueError('The output folder contains a different configuration.')
save_json(run_config, config_path)
df_model_counts = pd.DataFrame(model_count_rows)
display(df_model_counts)
print('Selected count(s):', [int(rate * SAMPLE_SIZE) for rate in SELECTION_RATES])
print('Output folder:', RUN_DIR)

## Run the three algorithms

This is the cell that performs the DP. The progress bar counts completed
models within each cohort. Cohorts and seeds run sequentially while models
run in parallel. Each saved prediction variant / threshold / selection-rate
combination has its own completion record. A completed combination is loaded
on rerun; an interrupted combination is recalculated.

The `people` and allocation CSVs use the same filenames and column formats as
the baseline results. To inspect them in the existing results notebook, set
its `ALLOCATION_DIR` to the printed `RUN_DIR` and use the same model class,
sample size, prediction variant, and selection rate.

In [ ]:
result_sets, timing_rows = [], []
prefixes = {'OPT': 'opt', 'LDA-tru': 'lda_tru', 'LDA-exp': 'lda_exp',
            'people': 'people', 'diagnostics': 'diagnostics'}
for pred_method in PRED_METHODS:
    for threshold in NEED_THRESHOLDS:
        frame = prediction_cache[(pred_method, threshold)]
        for rate in SELECTION_RATES:
            tag = allocation_tag(pred_method, threshold, rate)
            paths = {key: RUN_DIR / f'{prefix}_{tag}.csv' for key, prefix in prefixes.items()}
            completion_path = RUN_DIR / f'complete_{tag}.json'
            cached = completion_path.exists()
            if cached:
                completion = json.loads(completion_path.read_text())
                if completion.get('run_id') != run_id or completion.get('tag') != tag:
                    raise ValueError(f'Unexpected completion record: {completion_path}')
                for key, path in paths.items():
                    if not path.is_file() or file_hash(path) != completion['sha256'].get(key):
                        raise ValueError(f'Saved results are missing or changed: {path}')
                results = read_result_files(paths)
                elapsed = completion['elapsed_seconds']
                print('Loaded completed run:', tag)
            else:
                print('Running:', tag)
                allocation = make_allocator(frame, rate)
                started = perf_counter()
                opt, tru, exp, people = allocation.get_allocations(
                    exp_need_col=EXP_NEED_COL, algorithms=ALGORITHMS,
                )
                elapsed = perf_counter() - started
                results = {'OPT': opt, 'LDA-tru': tru, 'LDA-exp': exp, 'people': people,
                           'diagnostics': pd.DataFrame(allocation.allocation_diagnostics)}
                for key, result in results.items():
                    save_csv(result, paths[key])
                save_json({'run_id': run_id, 'tag': tag, 'elapsed_seconds': elapsed,
                           'sha256': {key: file_hash(path) for key, path in paths.items()}},
                          completion_path)
                print(f'Finished {len(people)} cohorts in {elapsed / 60:.2f} minutes.')
            metadata = {'pred_variant': pred_method, 'need_threshold': threshold,
                        'selection_rate': rate, 'n': SAMPLE_SIZE, 'k': int(rate * SAMPLE_SIZE)}
            result_sets.append({'metadata': metadata, 'results': results})
            timing_rows.append({**metadata, 'cohorts': len(results['people']),
                                'elapsed_seconds': elapsed, 'loaded_from_cache': cached})

df_timings = pd.DataFrame(timing_rows)
save_csv(df_timings, RUN_DIR / 'timings.csv')
display(df_timings)

## Compare allocations using true needs

Metrics are expanded to one observation per model and cohort, including
models that produce the same selection. The displayed ± is the **sample
standard deviation across these observations**, not a confidence interval.
Models sharing a cohort are not independent cohort observations.

Absolute disparity is the absolute difference between the two selected
groups' mean true needs. If a selected group is empty, disparity is undefined
and remains `NaN`; the summary reports how many such observations there are.
The disparity ratio is also undefined if either group mean is zero. Top-k
fallbacks are retained and reported rather than removed from the comparison.

In [ ]:
metric_rows, diagnostic_frames = [], []
for experiment in result_sets:
    metadata, results = experiment['metadata'], experiment['results']
    pred_method, threshold = metadata['pred_variant'], metadata['need_threshold']
    frame = prediction_cache[(pred_method, threshold)]
    seed_frames = {seed: split.loc[split.person_id >= 0].set_index('person_id')
                   for seed, split in frame.groupby('seed')}
    for cohort_info in results['people'].itertuples():
        cohort = seed_frames[cohort_info.seed].loc[cohort_info.people]
        if len(cohort) != SAMPLE_SIZE or not cohort.index.is_unique:
            raise ValueError('Saved cohort has the wrong size or duplicate people.')
        expected_models = set(eligible_models[(pred_method, threshold, cohort_info.seed)])
        for algorithm in ALGORITHMS:
            allocations = results[algorithm]
            subset = allocations.loc[(allocations.seed == cohort_info.seed)
                                     & (allocations.iteration == cohort_info.iteration)]
            observed_models = [model for models in subset.models for model in models]
            if set(observed_models) != expected_models or len(observed_models) != len(expected_models):
                raise ValueError(f'{algorithm}: missing or repeated eligible models in a cohort.')
            for row in subset.itertuples():
                if len(row.selected) != metadata['k'] or len(set(row.selected)) != metadata['k']:
                    raise ValueError('Saved allocation has the wrong selection count or duplicate people.')
                selected = cohort.loc[row.selected]
                needs_0 = selected.loc[selected[GROUP_COL] == 0, NEED_COL]
                needs_1 = selected.loc[selected[GROUP_COL] == 1, NEED_COL]
                mean_0, mean_1 = needs_0.mean(), needs_1.mean()
                disparity = abs(mean_1 - mean_0)
                ratio = max(mean_0, mean_1) / min(mean_0, mean_1) if mean_0 > 0 and mean_1 > 0 else np.nan
                for model in row.models:
                    metric_rows.append({**metadata, 'seed': cohort_info.seed,
                        'iteration': cohort_info.iteration, 'algorithm': algorithm, 'model': model,
                        'realized_utility': selected.y.mean(), 'expected_utility': selected[model].mean(),
                        'absolute_disparity': disparity, 'disparity_ratio': ratio,
                        'selected_count_0': len(needs_0), 'selected_count_1': len(needs_1)})
    diagnostic_frames.append(results['diagnostics'].assign(**metadata))

df_metrics = pd.DataFrame(metric_rows)
df_diagnostics = pd.concat(diagnostic_frames, ignore_index=True)
SUMMARY_KEYS = ['pred_variant', 'need_threshold', 'selection_rate', 'k', 'n', 'algorithm']
METRICS = ['realized_utility', 'expected_utility', 'absolute_disparity', 'disparity_ratio']
JOIN_KEYS = [*SUMMARY_KEYS, 'seed', 'iteration', 'model']
df_metrics = df_metrics.merge(df_diagnostics[JOIN_KEYS + ['fallback_used']],
                              on=JOIN_KEYS, how='left', validate='one_to_one')
if df_metrics.fallback_used.isna().any():
    raise ValueError('Some model allocations are missing their diagnostics.')

grouped = df_metrics.groupby(SUMMARY_KEYS, dropna=False, sort=False)
df_summary_numeric = grouped[METRICS].agg(['mean', 'std'])
df_summary_numeric.columns = ['_'.join(column) for column in df_summary_numeric.columns]
df_summary_numeric = df_summary_numeric.reset_index()
counts = grouped.agg(model_cohort_observations=('model', 'size'),
                     valid_disparity=('absolute_disparity', 'count'),
                     valid_ratio=('disparity_ratio', 'count'),
                     fallback_rate=('fallback_used', 'mean')).reset_index()
counts['undefined_disparity'] = counts.model_cohort_observations - counts.valid_disparity
cohort_counts = df_metrics[SUMMARY_KEYS + ['seed', 'iteration']].drop_duplicates()
cohort_counts = cohort_counts.groupby(SUMMARY_KEYS, dropna=False, sort=False).size().rename('cohorts').reset_index()
df_summary_numeric = df_summary_numeric.merge(counts, on=SUMMARY_KEYS).merge(cohort_counts, on=SUMMARY_KEYS)
df_summary_numeric['algorithm'] = pd.Categorical(df_summary_numeric.algorithm, ALGORITHM_ORDER, ordered=True)
df_summary_numeric = df_summary_numeric.sort_values(SUMMARY_KEYS).reset_index(drop=True)

def mean_sd(mean, std):
    if pd.isna(mean):
        return 'undefined'
    return f'{mean:.2f} ± {std:.2f}' if pd.notna(std) else f'{mean:.2f} (SD unavailable)'

df_summary = df_summary_numeric[SUMMARY_KEYS].copy()
for metric in METRICS:
    df_summary[metric] = [mean_sd(mean, std) for mean, std in
                          zip(df_summary_numeric[metric + '_mean'], df_summary_numeric[metric + '_std'])]
for column in ['cohorts', 'model_cohort_observations', 'valid_disparity', 'undefined_disparity', 'valid_ratio', 'fallback_rate']:
    df_summary[column] = df_summary_numeric[column]

df_cohort_summary = df_metrics.groupby(SUMMARY_KEYS + ['seed', 'iteration'], dropna=False)[METRICS].mean().reset_index()
save_csv(df_metrics, RUN_DIR / 'allocation_metrics.csv')
save_csv(df_summary_numeric, RUN_DIR / 'summary_numeric.csv')
save_csv(df_summary, RUN_DIR / 'summary_formatted.csv')
save_csv(df_cohort_summary, RUN_DIR / 'cohort_mean_metrics.csv')
display(df_summary.set_index(SUMMARY_KEYS))

## Absolute disparity with standard deviation

Bars show the mean disparity and error bars show ± one sample SD. The lower
error bar is clipped at zero. Undefined disparity observations are counted
in the table above and excluded from the mean/SD calculation. An SD is not
drawn when fewer than two valid observations are available.

In [ ]:
colors = {'LDA-exp': '#7560a8', 'LDA-tru': '#009e73', 'OPT': '#4a4a4a'}
light_colors = {'LDA-exp': 'rgba(117,96,168,0.45)',
                'LDA-tru': 'rgba(0,158,115,0.45)', 'OPT': 'rgba(74,74,74,0.4)'}
figures = []
plot_keys = ['pred_variant', 'need_threshold', 'selection_rate', 'k', 'n']
for setting, comparison in df_summary_numeric.groupby(plot_keys, observed=True, sort=False):
    pred_method, threshold, rate, k, n = setting
    fig = go.Figure()
    for row in comparison.itertuples():
        if pd.isna(row.absolute_disparity_mean):
            continue
        mean, std = row.absolute_disparity_mean, row.absolute_disparity_std
        error = (dict(type='data', symmetric=False, array=[std], arrayminus=[min(mean, std)],
                      color=light_colors[row.algorithm], thickness=2, width=7)
                 if pd.notna(std) else None)
        fig.add_trace(go.Bar(x=[row.algorithm], y=[mean], name=row.algorithm,
                             marker_color=colors[row.algorithm], error_y=error,
                             hovertemplate='%{x}<br>Mean absolute disparity: %{y:.4f}<extra></extra>'))
    fig.update_layout(template='plotly_white', showlegend=False, width=850, height=500,
                      title=f'Absolute disparity: n={n}, k={k}, {pred_method}, threshold={threshold}',
                      xaxis=dict(title='Algorithm', categoryorder='array', categoryarray=ALGORITHM_ORDER),
                      yaxis=dict(title='Absolute disparity in true needs', rangemode='tozero'))
    figure_path = RUN_DIR / f'absolute_disparity_{allocation_tag(pred_method, threshold, rate)}.html'
    fig.write_html(figure_path, include_plotlyjs=True)
    fig.show()
    figures.append(fig)

print('All allocations, people, diagnostics, comparison tables, and plots are saved in:')
print(RUN_DIR)